In [38]:
# Install required packages 
%pip install -U -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [55]:
import boto3
from datetime import datetime, date
from boto3.session import Session
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
import boto3, base64, json
from pathlib import Path

In [40]:
boto_session = Session()
REGION = boto_session.region_name
MODEL_ID = "us.amazon.nova-2-lite-v1:0"

In [41]:
import boto3

bedrock = boto3.client("bedrock", region_name="us-west-2")

# Check what Nova models are accessible
models = bedrock.list_foundation_models(byOutputModality="TEXT")
for m in models["modelSummaries"]:
    if "nova" in m["modelId"].lower() or "claude" in m["modelId"].lower():
        print(m["modelId"], "|", m.get("inferenceTypesSupported", []))

anthropic.claude-sonnet-4-20250514-v1:0 | ['INFERENCE_PROFILE']
anthropic.claude-haiku-4-5-20251001-v1:0 | ['INFERENCE_PROFILE']
anthropic.claude-sonnet-4-6 | ['INFERENCE_PROFILE']
amazon.nova-pro-v1:0 | ['INFERENCE_PROFILE']
amazon.nova-2-lite-v1:0 | ['INFERENCE_PROFILE']
amazon.nova-2-sonic-v1:0 | ['ON_DEMAND']
anthropic.claude-opus-4-6-v1 | ['INFERENCE_PROFILE']
anthropic.claude-opus-4-7 | ['INFERENCE_PROFILE']
anthropic.claude-sonnet-4-5-20250929-v1:0 | ['INFERENCE_PROFILE']
anthropic.claude-opus-4-1-20250805-v1:0 | ['INFERENCE_PROFILE']
anthropic.claude-opus-4-5-20251101-v1:0 | ['INFERENCE_PROFILE']
amazon.nova-premier-v1:0:8k | []
amazon.nova-premier-v1:0:20k | []
amazon.nova-premier-v1:0:1000k | []
amazon.nova-premier-v1:0:mm | []
amazon.nova-premier-v1:0 | ['INFERENCE_PROFILE']
amazon.nova-lite-v1:0 | ['INFERENCE_PROFILE']
amazon.nova-micro-v1:0 | ['INFERENCE_PROFILE']
anthropic.claude-3-5-haiku-20241022-v1:0 | ['ON_DEMAND']
anthropic.claude-3-sonnet-20240229-v1:0:28k | ['PROVI

In [42]:
@tool
def extract_document_metadata(document_text: str) -> str:
    doc_text_lower = document_text.lower()

    if "basic life support" in doc_text_lower or "bls" in doc_text_lower:
        doc_type = "BLS"
    elif "advanced cardiac life support" in doc_text_lower or "acls" in doc_text_lower:
        doc_type = "ACLS"
    elif "tuberculosis" in doc_text_lower or "tb test" in doc_text_lower or "ppd" in doc_text_lower:
        doc_type = "TB_TEST"
    elif "resume" in doc_text_lower or "curriculum vitae" in doc_text_lower:
        doc_type = "RESUME"
    elif "assessment" in doc_text_lower or "competency exam" in doc_text_lower:
        doc_type = "ASSESSMENT"
    else:
        doc_type = "UNKNOWN"

    return f"Document type detected: {doc_type}\nRaw text length: {len(document_text)} chars"

In [43]:
@tool
def check_compliance_rules(doc_type: str, expiry_date: str = "", 
                            issuer: str = "", score: int = -1,
                            test_date: str = "") -> str:
    """
    Apply compliance rules for a given document type.
    Args:
        doc_type: One of BLS, ACLS, TB_TEST, RESUME, ASSESSMENT
        expiry_date: Expiry date string (YYYY-MM-DD) for certs
        issuer: Issuing organization name
        score: Numeric score for assessments (0-100), -1 if N/A
        test_date: Test/completion date for TB test (YYYY-MM-DD)
    Returns:
        Pass/fail for each rule with details
    """
    today = date.today()
    results = []

    if doc_type == "BLS":
        # Rule 1: Not expired
        if expiry_date:
            exp = date.fromisoformat(expiry_date)
            results.append(f"Expiry check: {'PASS' if exp >= today else 'FAIL'} (expires {expiry_date})")
        else:
            results.append("Expiry check: FAIL — no expiry date found")
        # Rule 2: Accredited issuer
        valid_issuers = ["american heart association", "aha", "american red cross", "red cross"]
        issuer_ok = any(v in issuer.lower() for v in valid_issuers)
        results.append(f"Issuer check: {'PASS' if issuer_ok else 'FAIL'} (issuer: {issuer or 'not found'})")

    elif doc_type == "ACLS":
        if expiry_date:
            exp = date.fromisoformat(expiry_date)
            results.append(f"Expiry check: {'PASS' if exp >= today else 'FAIL'} (expires {expiry_date})")
        else:
            results.append("Expiry check: FAIL — no expiry date found")
        # AHA only for ACLS
        issuer_ok = "american heart association" in issuer.lower() or "aha" in issuer.lower()
        results.append(f"Issuer check: {'PASS' if issuer_ok else 'FAIL'} (issuer: {issuer or 'not found'})")

    elif doc_type == "TB_TEST":
        # Must be within 1 year
        if test_date:
            td = date.fromisoformat(test_date)
            days_elapsed = (today - td).days
            results.append(f"Recency check: {'PASS' if days_elapsed <= 365 else 'FAIL'} ({days_elapsed} days ago)")
        else:
            results.append("Recency check: FAIL — no test date found")

    elif doc_type == "RESUME":
        # Checked by agent reasoning, not a numeric rule
        results.append("Rule: Agent will verify required credentials are present")
        results.append("Required credentials: active nursing/clinical license, BLS cert, relevant experience")

    elif doc_type == "ASSESSMENT":
        if score >= 0:
            results.append(f"Score check: {'PASS' if score >= 80 else 'FAIL'} (score: {score}/100, threshold: 80)")
        else:
            results.append("Score check: FAIL — no score found")

    else:
        results.append("UNKNOWN document type — cannot apply rules")

    return "\n".join(results)

In [44]:
@tool
def flag_risk_patterns(document_text: str, doc_type: str) -> str:
    """
    Scan for red flags: missing signatures, suspicious issuers, formatting issues.
    Args:
        document_text: Raw text of the document
        doc_type: Document type for context
    Returns:
        List of detected risk flags, or 'No flags found'
    """
    flags = []
    text_lower = document_text.lower()

    if "signature" not in text_lower and doc_type in ["BLS", "ACLS", "TB_TEST"]:
        flags.append("WARNING: No signature detected")
    if "expired" in text_lower:
        flags.append("WARNING: Document contains the word 'expired'")
    if doc_type in ["BLS", "ACLS"] and "online only" in text_lower:
        flags.append("WARNING: Online-only certification — AHA requires hands-on skills check")
    if len(document_text.strip()) < 100:
        flags.append("WARNING: Document is unusually short — may be incomplete")

    return "\n".join(flags) if flags else "No risk flags detected"

In [45]:
@tool
def lookup_prior_decisions(doc_type: str, query: str) -> str:
    """
    Search AgentCore Memory for similar past compliance decisions.
    Args:
        doc_type: Document type to scope the search
        query: What to search for (e.g. 'expired BLS from Red Cross')
    Returns:
        Relevant past decisions retrieved from memory
    """
    return f"[Memory lookup for {doc_type}: '{query}'] — no prior decisions seeded yet. Will populate after Lab 2."

In [52]:
SYSTEM_PROMPT = """You are a healthcare workforce compliance reviewer.

Your job is to review submitted documents and issue a clear APPROVE or REJECT decision.

For every document you receive:
1. Call extract_document_metadata to identify the document type
2. Call check_compliance_rules with the relevant fields (expiry, issuer, score, test date)
3. Call flag_risk_patterns to check for red flags
4. Call lookup_prior_decisions to check if similar cases have been reviewed before
5. Based on all tool results, issue a final decision

Your response MUST always end with exactly these three lines:
DECISION: APPROVE or REJECT
REASON: One sentence explaining the primary reason
RULES CHECKED: List the rules that passed and failed

For RESUME documents: APPROVE if active license + valid BLS cert present. REJECT if either is missing.
For ASSESSMENT documents: APPROVE if score >= 80. REJECT if score < 80 or not found.

Be strict. When in doubt, REJECT and explain what is missing.
You must always output the DECISION line — never leave it blank.

IMPORTANT: After calling all tools, you MUST write your final answer as plain text.
Never end your response with a tool call. Always output the DECISION line last.
"""

In [47]:
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

agent = Agent(
    model=model,
    tools=[extract_document_metadata, check_compliance_rules, 
           flag_risk_patterns, lookup_prior_decisions],
    system_prompt=SYSTEM_PROMPT,
)

In [48]:
import uuid
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

REVIEWER_ID = "compliance-reviewer-001"

# ── Create memory with two strategies ─────────────────────────────────────
memory_manager = MemoryManager(region_name=REGION)
memory = memory_manager.get_or_create_memory(
    name="ComplianceMemory",
    strategies=[
        {
            StrategyType.SEMANTIC.value: {
                "name": "PastDecisions",
                "description": "Stores past compliance decisions and their rationale",
                "namespaces": ["compliance/decisions/{actorId}/semantic/"],
            }
        },
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "ReviewerPatterns",
                "description": "Captures reviewer patterns and strictness preferences",
                "namespaces": ["compliance/reviewer/{actorId}/preferences/"],
            }
        },
    ]
)
memory_id = memory["id"]
print(f"Memory ID: {memory_id}")

# ── Seed past decisions so memory is useful from day 1 ────────────────────
memory_client = MemoryClient(region_name=REGION)

past_decisions = [
    ("BLS cert from American Red Cross, expires 2026-03-15, signed by instructor.", "USER"),
    ("APPROVED. Expiry valid, Red Cross is accredited issuer, signature present.", "ASSISTANT"),

    ("ACLS cert issued by online-only provider 'CertifyNow', expiry 2025-12-01.", "USER"),
    ("REJECTED. AHA requires hands-on skills check — online-only ACLS not accepted.", "ASSISTANT"),

    ("TB test dated 14 months ago, result negative.", "USER"),
    ("REJECTED. TB test must be within 12 months. This one is 14 months old.", "ASSISTANT"),

    ("Resume with active RN license, current BLS, 3 years ICU experience.", "USER"),
    ("APPROVED. All required credentials present and verifiable.", "ASSISTANT"),

    ("Competency assessment score: 74/100.", "USER"),
    ("REJECTED. Minimum passing score is 80. Score of 74 does not meet threshold.", "ASSISTANT"),
]

memory_client.create_event(
    memory_id=memory_id,
    actor_id=REVIEWER_ID,
    session_id="seed-session-001",
    messages=past_decisions,
)
print("Seeded past decisions into memory")

# ── Wire memory into the agent ─────────────────────────────────────────────
memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=str(uuid.uuid4()),
    actor_id=REVIEWER_ID,
    retrieval_config={
        "compliance/decisions/{actorId}/semantic/": RetrievalConfig(top_k=3, relevance_score=0.3),
        "compliance/reviewer/{actorId}/preferences/": RetrievalConfig(top_k=2, relevance_score=0.2),
    }
)

agent = Agent(
    model=model,
    session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
    tools=[extract_document_metadata, check_compliance_rules,
           flag_risk_patterns, lookup_prior_decisions],
    system_prompt=SYSTEM_PROMPT,
)

✅ MemoryManager initialized for region: us-west-2
Memory already exists. Using existing memory ID: ComplianceMemory-4NJ8B83zlG
🔎 Retrieving memory resource with ID: ComplianceMemory-4NJ8B83zlG...
  Found memory: ComplianceMemory-4NJ8B83zlG
Existing {'type': 'SEMANTIC', 'name': 'PastDecisions', 'description': 'Stores past compliance decisions and their rationale', 'namespaces': ['compliance/decisions/{actorId}/semantic/']}
Requested {'type': 'SEMANTIC', 'name': 'PastDecisions', 'description': 'Stores past compliance decisions and their rationale', 'namespaces': ['compliance/decisions/{actorId}/semantic/']}
Existing {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'description': 'Captures reviewer patterns and strictness preferences', 'namespaces': ['compliance/reviewer/{actorId}/preferences/']}
Requested {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'description': 'Captures reviewer patterns and strictness preferences', 'namespaces': ['compliance/reviewer/{actorId}/pref

Memory ID: ComplianceMemory-4NJ8B83zlG
Seeded past decisions into memory


In [54]:
# ── Synthetic test documents ───────────────────────────────────────────────

TB_01_TEXT = """
TUBERCULOSIS SCREENING RECORD

Facility: Riverside Community Health Clinic
Provider: Dr. Anita Patel, MD

Patient Name: Luis Fernandez
Date of Test (PPD placed): 2026-01-10
Date of Reading: 2026-01-13
Result: NEGATIVE — Induration: 0mm

Interpretation: No evidence of TB infection.
Cleared for: Healthcare employment

Physician Signature: Dr. A. Patel
License No.: IL-MD-88210
"""

TB_02_TEXT = """
TUBERCULOSIS (TB) SKIN TEST RESULT

Clinic: Northside Occupational Health
Administering Nurse: R. Cho, RN

Employee Name: Sandra Kimura
PPD Administered: 2024-10-05
Date of Reading: 2024-10-07
Result: NEGATIVE — 0mm induration

This record certifies a negative TB skin test result.
Recommended retesting: Annually or per employer policy.
"""

ACLS_01_TEXT = """
ADVANCED CARDIOVASCULAR LIFE SUPPORT (ACLS)
PROVIDER CERTIFICATION

Issuing Organization: American Heart Association
Course Type: ACLS Provider — In-Person

Provider Name: Dr. Kevin Tran, MD
Date Completed: 2024-06-20
Expiration Date: 2026-06-20

Skills Assessed:
- Systematic approach to ACLS cases
- Megacode scenario: PASSED
- Rhythm recognition: PASSED
- Team dynamics and communication: PASSED

Medical Director Signature: Dr. L. Okafor
Site ID: AHA-NY-3310

Renewal required every 2 years.
"""

ACLS_02_TEXT = """
ACLS CERTIFICATION OF COMPLETION

Provider: NationalCertifyNow LLC
Delivery Method: 100% Online — No in-person component

Student Name: Angela Pierce
Date Issued: 2025-01-10
Valid Through: 2027-01-10

This certificate is awarded upon successful completion of the online ACLS
knowledge assessment. Megacode and hands-on skills verification are not
included in this course format.

NationalCertifyNow LLC is not affiliated with the American Heart Association.
"""

RESUME_01_TEXT = """
RESUME — CLINICAL STAFF APPLICATION

Name: Patricia Nguyen, RN, BSN
License: Registered Nurse — Illinois License No. RN-204851 (Active, expires 2026-05-31)

Certifications:
- BLS: American Heart Association, valid through 2026-03-01
- ACLS: American Heart Association, valid through 2025-11-15

Education:
- BSN, DePaul University, 2018

Experience:
- 4 years, Medical-Surgical Unit, Northwestern Memorial Hospital (2019–2023)
- 2 years, Cardiac Step-Down Unit, Rush University Medical Center (2023–present)

References available upon request.
"""

RESUME_02_TEXT = """
APPLICATION FOR NURSING POSITION

Applicant: Marcus Webb
Nursing School: Graduated 2021 — Associates Degree in Nursing, City College

Certifications: CPR trained (community class, 2019)

Work History:
- Patient Care Technician, Metro Hospital, 2021–2022
- Home Health Aide, CareFirst Agency, 2022–2024

Note: Currently studying for NCLEX. License application pending.
No current BLS certification on file.
"""

ASSESSMENT_01_TEXT = """
CLINICAL COMPETENCY ASSESSMENT — RESULTS

Assessment Title: Medical-Surgical Nursing Competency Evaluation
Administered by: TalentPlus Healthcare Staffing
Date Administered: 2025-03-22

Candidate: Rachel Kim
Employee ID: TP-20291

Sections:
  Patient Safety & Fall Prevention: 94/100
  Medication Administration: 88/100
  Wound Care & Infection Control: 91/100
  Documentation & Charting: 85/100

Overall Score: 89.5 / 100
Pass Threshold: 80 / 100
Result: PASS

Reviewed by: Clinical Education Team
"""

ASSESSMENT_02_TEXT = """
COMPETENCY ASSESSMENT RECORD

Program: ICU Nursing Skills Assessment
Organization: Midwest Staffing Solutions
Assessment Date: 2025-04-01

Candidate Name: Jerome Patterson

Module Results:
  Hemodynamic Monitoring: 65/100
  Ventilator Management: 72/100
  Critical Drip Titration: 70/100
  Emergency Response Protocols: 68/100

Composite Score: 68.75 / 100
Minimum Passing Score: 80 / 100
Status: DID NOT PASS

Candidate may retest after 30-day remediation period.
"""

# ── Test cases list ────────────────────────────────────────────────────────

test_cases = [
    {"id": "BLS-01",        "input": BLS_01_TEXT,        "expected_decision": "APPROVE", "doc_type": "BLS"},
    {"id": "BLS-02",        "input": BLS_02_TEXT,        "expected_decision": "REJECT",  "doc_type": "BLS"},
    {"id": "ACLS-01",       "input": ACLS_01_TEXT,       "expected_decision": "APPROVE", "doc_type": "ACLS"},
    {"id": "ACLS-02",       "input": ACLS_02_TEXT,       "expected_decision": "REJECT",  "doc_type": "ACLS"},
    {"id": "TB-01",         "input": TB_01_TEXT,         "expected_decision": "APPROVE", "doc_type": "TB_TEST"},
    {"id": "TB-02",         "input": TB_02_TEXT,         "expected_decision": "REJECT",  "doc_type": "TB_TEST"},
    {"id": "RESUME-01",     "input": RESUME_01_TEXT,     "expected_decision": "APPROVE", "doc_type": "RESUME"},
    {"id": "RESUME-02",     "input": RESUME_02_TEXT,     "expected_decision": "REJECT",  "doc_type": "RESUME"},
    {"id": "ASSESSMENT-01", "input": ASSESSMENT_01_TEXT, "expected_decision": "APPROVE", "doc_type": "ASSESSMENT"},
    {"id": "ASSESSMENT-02", "input": ASSESSMENT_02_TEXT, "expected_decision": "REJECT",  "doc_type": "ASSESSMENT"},
]

import re

def get_response_text(response):
    try:
        content = response.message["content"]
        for block in content:
            if isinstance(block, dict) and "text" in block:
                return block["text"]
        return str(response)
    except Exception:
        return str(response)

def make_fresh_agent():
    return Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
        tools=[extract_document_metadata, check_compliance_rules,
               flag_risk_patterns, lookup_prior_decisions],
        system_prompt=SYSTEM_PROMPT,
    )

results = []

for tc in test_cases:
    print(f"\nRunning {tc['id']} ({tc['doc_type']}) — expected: {tc['expected_decision']}")
    
    fresh_agent = make_fresh_agent()          # ← fresh agent every time
    response = fresh_agent(tc["input"])
    response_text = get_response_text(response)  # ← safe extractor

    print(f"  Raw: {response_text[:150]}")    # ← always print so you can see what's happening

    if re.search(r"DECISION\s*:\s*APPROVE", response_text, re.IGNORECASE):
        actual = "APPROVE"
    elif re.search(r"DECISION\s*:\s*REJECT", response_text, re.IGNORECASE):
        actual = "REJECT"
    else:
        actual = "UNCLEAR"
        print(f"  ⚠️ Full response: {response_text}")

    correct = actual == tc["expected_decision"]
    results.append({
        "id": tc["id"], "doc_type": tc["doc_type"],
        "expected": tc["expected_decision"], "actual": actual,
        "correct": correct, "response": response_text,
    })
    print(f"  {'✅' if correct else '❌'}  Got: {actual}")

# Summary
print("\n" + "="*50)
print("EVAL RESULTS SUMMARY")
print("="*50)

total   = len(results)
correct = sum(r["correct"] for r in results)

for r in results:
    status = "✅ PASS" if r["correct"] else "❌ FAIL"
    print(f"  {status}  {r['id']:15s}  expected={r['expected']:7s}  got={r['actual']}")

print(f"\nAccuracy: {correct}/{total} = {correct/total:.0%}")

print("\nBy doc type:")
for doc_type in ["BLS", "ACLS", "TB_TEST", "RESUME", "ASSESSMENT"]:
    subset = [r for r in results if r["doc_type"] == doc_type]
    n_correct = sum(r["correct"] for r in subset)
    print(f"  {doc_type:12s}: {n_correct}/{len(subset)}")


Running BLS-01 (BLS) — expected: APPROVE

Tool #1: extract_document_metadata

Tool #2: check_compliance_rules

Tool #3: flag_risk_patterns

Tool #4: lookup_prior_decisions
DECISION: APPROVE
REASON: BLS certificate is active and valid through 2026-09-15 with proper issuer
RULES CHECKED: Expiry check: PASS (expires 2026-09-15), Issuer check: PASS (issuer: American Heart Association)  Raw: DECISION: APPROVE
REASON: BLS certificate is active and valid through 2026-09-15 with proper issuer
RULES CHECKED: Expiry check: PASS (expires 2026-09
  ✅  Got: APPROVE

Running BLS-02 (BLS) — expected: REJECT

Tool #1: extract_document_metadata

Tool #2: check_compliance_rules

Tool #3: flag_risk_patterns

Tool #4: lookup_prior_decisions
DECISION: REJECT
REASON: Expired certification and unrecognized issuer.
RULES CHECKED: Expiry check: FAIL (expires 2024-11-03), Issuer check: FAIL (issuer: ProMed Online Certifications)  Raw: DECISION: REJECT
REASON: Expired certification and unrecognized issuer.
RULE